# Preprocessing from raw traces: LOS/NLOS classification (Argos massive-MIMO dataset)

This notebook reads the **raw I/Q samples** in `raw_traces/*.hdf5`, extracts the real CSI from them, and prepares the data for the four models in the paper: SVM, RF, CNN-LSTM and CNN-Transformer.

**How this differs from the paper's preprocessing:** the paper averages |I/Q| over all 1120 raw samples, which gives roughly the received signal level per antenna. Here the CSI is extracted properly:
pilot samples → FFT → 52 subcarriers → calibration. This follows the dataset authors' own code (`tvt_open_source/lib/argos_lib.py` and `lib/ota_lib.py`).

**Normalisation per antenna (default):** each antenna's 52 subcarrier amplitudes are divided by that antenna's own mean. This removes the pattern of which antennas are stronger, which depends on where the user stands relative to the array. With only per-frame normalisation, the SVM learned that location pattern and misclassified the whole of LOS cluster 3 (0% accuracy). Per-antenna normalisation keeps the frequency shape (multipath), which is what separates LOS from NLOS.

**Full resolution:** by default (`FEATURE = "full"`) no averaging is done. Every frame keeps all 64 antennas × 52 subcarriers = **3328 amplitude values**, and the complex CSI (amplitude and phase) is also saved for every cluster. All 9 clusters are processed. The only frames dropped are the first 200 of each cluster (receiver gain settling) and the last one (all zeros); set `SKIP_FIRST = 0` to keep the settling frames as well.

| Output | Approx. size |
|---|---|
| frame datasets (train + test) | 0.7 GB |
| sequence datasets (train + test) | 3.5 GB |
| per-cluster features | 0.7 GB |
| per-cluster complex CSI | 1.4 GB |
| **total** | **about 6.3 GB** |

The large arrays are written straight to disk (memory-mapped `.npy`), so the notebook peaks at about 4 GB of RAM (measured), not 6 GB or more.

**Steps**
1. Setup and configuration
2. Inspect a raw file
3. Understand the raw sample layout
4. CSI extraction function (pilot → FFT → subcarriers → antennas → calibration)
5. Try it on one cluster
6. Amplitude, normalisation and feature reduction
7. Quality check
8. Process all 9 clusters
9. Save each cluster separately (features, and optionally the full CSI)
10. z-score with training-only statistics
11. Frame-level datasets (SVM / RF)
12. Sliding-window sequences (CNN-LSTM / CNN-Transformer)
13. Save outputs
14. Final checks (including a comparison with `channel_data/`)

## Step 1: Setup and configuration

**Running in Colab:** uncomment the Drive lines and set `RAW_DIR` to the `raw_traces` folder on your Drive.

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')
# RAW_DIR = "/content/drive/MyDrive/Project/Project1/raw_data/renew_dataset/tvt_open_source/raw_traces"
# OUT_DIR = "/content/drive/MyDrive/Project/Project1/processed_raw_full"

import os
import json

import h5py
import numpy as np

RAW_DIR = "../tvt_open_source/raw_traces"
OUT_DIR = "output_raw_full"

FEATURE = "full"      # "full" (3328 = 64 ant x 52 sc, no reduction) | "both" (116) | "subcarrier" (52) | "antenna" (64)
NORM = "antenna"      # "antenna" = each antenna / its own mean (recommended; removes location-dependent array pattern)
                      # "frame"   = each frame / its mean (keeps the spatial pattern; LOS 3 failed with this)
                      # "none"    = absolute amplitude (paper-style)
WINDOW = 50
STRIDE = 10
SKIP_FIRST = 200      # first frames are discarded (receiver gain still settling)
CHUNK = 500           # frames read from disk at a time
SAVE_CSI = True       # also save full complex CSI (amplitude + phase) per cluster (~1.4 GB for all 9)
CLUSTER_DIR = os.path.join(OUT_DIR, "clusters")

# Cluster-wise split, identical to the paper (derived from Table 4 / sequence counts)
TRAIN_CLUSTERS = ["LOS_cluster1", "LOS_cluster2",
                  "NLOS_cluster1", "NLOS_cluster2", "NLOS_cluster3"]
TEST_CLUSTERS = ["LOS_cluster3", "LOS_cluster4",
                 "NLOS_cluster4", "NLOS_cluster5"]

def label_of(cluster):
    return 0 if cluster.startswith("NLOS") else 1  # LOS = 1, NLOS = 0

def file_of(cluster):
    return os.path.join(RAW_DIR, f"channe14_{cluster}.hdf5")

if NORM not in ("antenna", "frame", "none"):
    raise ValueError(f"unknown NORM: {NORM}")
if NORM == "antenna" and FEATURE in ("antenna", "both"):
    # after per-antenna normalisation every antenna's mean is exactly 1, so the
    # 64 antenna-average features would be constant and carry no information
    raise ValueError('NORM = "antenna" removes the antenna averages; use FEATURE = "full" or "subcarrier"')

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CLUSTER_DIR, exist_ok=True)

## Step 2: Inspect a raw file

Only `Pilot_Samples` is needed. Its shape is (frames, 72 BS radio channels, 1120 samples, 2 = I/Q), stored as int16.
The file attributes describe how the 1120 samples are organised.

In [2]:
with h5py.File(file_of("LOS_cluster1"), "r") as f:
    for k in f.keys():
        print(f"{k:18s} {f[k].shape}  {f[k].dtype}")
    print()
    for a in ["num_bs_ant", "num_bs_nodes", "num_mob_ant", "samples_per_user",
              "sample_dump", "pilot_length", "channel", "total_num_frames"]:
        print(f"{a:18s} {f.attrs[a]}")

FrameCompleteTime  (8140, 2)  uint64
Pilot_Samples      (8140, 72, 1120, 2)  int16
RSSI               (8140, 5, 72)  uint32
mob_tx_gain        (62, 4, 2)  int32

num_bs_ant         72
num_bs_nodes       18
num_mob_ant        4
samples_per_user   224
sample_dump        1120
pilot_length       64
channel            14
total_num_frames   8140


## Step 3: Understand the raw sample layout

In each frame, each of the 72 radio channels records **1120 time samples**:

```
1120 = 5 slots x 224 samples   (samples_per_user = 224)
slot 0..3 : pilots from the 4 mobile antennas (num_mob_ant = 4)
slot 4    : noise only (nobody transmits)
```

Inside each 224-sample slot, the 802.11 **long training symbol (LTS)** starts at offset 47 and lasts 64 samples. A 64-point FFT of those samples, multiplied by the known LTS, gives the channel on each subcarrier.

Of the 4 mobile antennas, one is the **measured user** and one is the **calibration node**. The calibration node is used to remove phase offsets between the base-station radios:

| Clusters | measured slot | calibration slot |
|---|---|---|
| LOS | 0 | 3 |
| NLOS | 3 | 0 |

**Antennas:** 72 radio channels = 18 boards × 4. The dataset code reverses the antenna order, then drops the first 4 channels and 4 channels on a board that failed. That leaves **64 antennas** (the 8 × 8 array). This selection comes from the dataset authors; the dropped channels do not look faulty in the raw power.

**Checked on the raw data** (see `AUDIT.md`):
- Slot 4 is about 20 dB below the pilot slots, so it is noise only.
- The LTS actually starts at sample 46–54. Offset 47 stays inside the 32-sample cyclic prefix, so the amplitude is unaffected; only a linear phase ramp is added.
- In NLOS files, slots 0/1 are strongest and start at the same sample in every cluster, which fits a fixed calibration node near the base station. In LOS files, slots 2/3 behave that way.

In [3]:
# 802.11 LTS in the frequency domain (64 bins, fftshift order) -- from lib/argos_lts.py
LTS_FREQ = np.array([0,0,0,0,0,0,1,1,-1,-1,1,1,-1,1,-1,1,1,1,1,1,1,-1,-1,1,1,-1,1,-1,1,1,1,1,
                     0,1,-1,-1,1,1,-1,1,-1,1,-1,-1,-1,-1,-1,1,1,-1,-1,1,-1,1,-1,1,1,1,1,0,0,0,0,0], dtype=np.float32)
NULL_SUBCARRIERS = [0, 1, 2, 3, 4, 5, 32, 59, 60, 61, 62, 63]   # guard bands + DC
LTS_OFFSET = 47
FFT_SIZE = 64
ANT_IDX = np.delete(np.arange(4, 72), [32, 33, 34, 35])        # 64 antennas kept

def slots_for(cluster):
    # (measured slot, calibration slot)
    return (0, 3) if cluster.startswith("LOS") else (3, 0)

print("data subcarriers:", FFT_SIZE - len(NULL_SUBCARRIERS))
print("antennas kept:", len(ANT_IDX))

data subcarriers: 52
antennas kept: 64


## Step 4: CSI extraction function

For each chunk of frames and for both the measured slot and the calibration slot:

1. take the 64 LTS samples: `slot * 224 + 47` to `+ 64`
2. build complex values and scale to ±1: $x = (I + jQ) \cdot 2^{-15}$
3. run the FFT, shift it, and multiply by the known LTS: $H_k = \mathrm{FFT}(x)_k \cdot \mathrm{LTS}_k$
4. drop the 12 null subcarriers, leaving 52
5. reverse the antenna order and keep 64 antennas

Then **calibrate**: $H = H_{meas} \,/\, \dfrac{H_{cal}}{|H_{cal}|}$.
This changes only the phase, not the amplitude.

The first `SKIP_FIRST` frames (gain settling) and the last frame (all zeros) are dropped.

In [4]:
def extract_csi(path, cluster, chunk=CHUNK, skip_first=SKIP_FIRST):
    meas_slot, cal_slot = slots_for(cluster)
    with h5py.File(path, "r") as f:
        ds = f["Pilot_Samples"]                       # (frames, 72, 1120, 2) int16
        spu = int(f.attrs["samples_per_user"])        # 224
        n_frames = ds.shape[0]
        out = []
        for i in range(skip_first, n_frames - 1, chunk):
            block = ds[i:min(i + chunk, n_frames - 1)]
            per_slot = []
            for slot in (meas_slot, cal_slot):
                s = slot * spu + LTS_OFFSET
                iq = (block[:, :, s:s + FFT_SIZE, 0]
                      + 1j * block[:, :, s:s + FFT_SIZE, 1]).astype(np.complex64) * 2 ** -15
                h = np.fft.fftshift(np.fft.fft(iq, FFT_SIZE, axis=2), axes=2) * LTS_FREQ
                h = np.delete(h, NULL_SUBCARRIERS, axis=2)       # (c, 72, 52)
                h = h[:, ::-1, :][:, ANT_IDX, :]                 # (c, 64, 52)
                per_slot.append(h)
            h_meas, h_cal = per_slot
            out.append(h_meas / (h_cal / np.abs(h_cal)))
    return np.concatenate(out)                       # (frames, 64, 52) complex64

## Step 5: Try it on one cluster

In [5]:
H_demo = extract_csi(file_of("NLOS_cluster2"), "NLOS_cluster2")
print("CSI shape:", H_demo.shape, H_demo.dtype)
print("mean |H|:", np.abs(H_demo).mean())

CSI shape: (3853, 64, 52) complex64
mean |H|: 1.3630301


## Step 6: Amplitude, normalisation and feature reduction

1. **Amplitude:** $|H| = \sqrt{\mathrm{Re}(H)^2 + \mathrm{Im}(H)^2}$, shape (frames, 64, 52).
2. **Normalisation** (`NORM`):

   | `NORM` | Operation | Removes | Keeps |
   |---|---|---|---|
   | `"antenna"` (default) | each antenna ÷ mean of its 52 subcarriers | absolute power **and** the spatial pattern across antennas | frequency shape on every antenna (multipath) |
   | `"frame"` | each frame ÷ mean of all 3328 values | absolute power | spatial pattern + frequency shape |
   | `"none"` | — | nothing | everything, including absolute power |

   **Why `"antenna"` is the default:** which antennas are stronger depends on the user's direction relative to the 8 × 8 array, not on LOS/NLOS. With `"frame"`, the SVM learned this location pattern. LOS cluster 3 lies in a direction not seen in training: its antenna pattern correlates −0.39 to −0.47 with the other LOS clusters, so it was classified 0% correctly. With `"antenna"`, the same SVM reached 100% on LOS 3 and 99.4% overall (single split).

   Absolute power must also be removed, because the base station used 35 dB receive gain for every LOS file and 65 dB for every NLOS file.
   **No option removes the SNR difference** (about 38 dB for LOS vs 19 dB for NLOS). See `AUDIT.md`.
3. **Feature reduction** turns each frame into one vector:
   - `subcarrier`: mean over antennas → 52 values (frequency selectivity)
   - `antenna`: mean over subcarriers → 64 values (spatial pattern)
   - `both`: both of the above → 116 values
   - `full`: every antenna × subcarrier value → 3328 values

In [6]:
def frame_features(H, feature=FEATURE, norm=NORM):
    amp = np.abs(H).astype(np.float32)                                  # (frames, 64, 52)
    if norm == "antenna":
        amp = amp / (amp.mean(axis=2, keepdims=True) + 1e-8)            # per antenna, over subcarriers
    elif norm == "frame":
        amp = amp / (amp.mean(axis=(1, 2), keepdims=True) + 1e-8)      # per frame, over everything

    if feature == "subcarrier":
        return amp.mean(axis=1)
    if feature == "antenna":
        return amp.mean(axis=2)
    if feature == "both":
        return np.concatenate([amp.mean(axis=1), amp.mean(axis=2)], axis=1)
    if feature == "full":
        return amp.reshape(len(amp), -1)
    raise ValueError(f"unknown feature: {feature}")

feats_demo = frame_features(H_demo)
print("features shape:", feats_demo.shape)

features shape: (3853, 3328)


## Step 7: Quality check

This step removes frames with NaN or infinite values (for example, when the calibration slot is exactly zero) and frames that are all zeros.

In [7]:
def check_quality(name, x):
    bad = ~np.isfinite(x)
    zero_frames = np.all(x == 0, axis=1)
    if bad.any() or zero_frames.any():
        print(f"  WARNING {name}: {bad.sum()} non-finite values, "
              f"{zero_frames.sum()} all-zero frames (removed)")
    keep = ~(bad.any(axis=1) | zero_frames)
    return x[keep]

feats_demo = check_quality("NLOS_cluster2", feats_demo)
print("after check:", feats_demo.shape)
del H_demo, feats_demo

after check: (3853, 3328)


## Step 8: Process all 9 clusters

Steps 4–7 are applied to every raw file. The mean |H| *before* normalisation is recorded so you can compare received power between clusters.

If `SAVE_CSI = True`, the full complex CSI of each cluster, with shape (frames, 64, 52), is also saved to `clusters/<cluster>_csi.npy`. That is about 1.4 GB for all 9 clusters. It is needed only for the physical analysis (K-factor, eigenvalues, correlation) or for trying other features later.

In [8]:
per_cluster, raw_level = {}, {}

for c in TRAIN_CLUSTERS + TEST_CLUSTERS:
    H = extract_csi(file_of(c), c)
    raw_level[c] = float(np.abs(H).mean())
    per_cluster[c] = check_quality(c, frame_features(H))
    if SAVE_CSI:
        np.save(os.path.join(CLUSTER_DIR, f"{c}_csi.npy"), H.astype(np.complex64))
    print(f"{c:14s} label={label_of(c)}  frames={len(per_cluster[c]):5d}  "
          f"features={per_cluster[c].shape[1]}  mean |H|={raw_level[c]:.3f}")
    del H

LOS_cluster1   label=1  frames= 7939  features=3328  mean |H|=1.849
LOS_cluster2   label=1  frames= 7958  features=3328  mean |H|=1.792
NLOS_cluster1  label=0  frames= 5880  features=3328  mean |H|=1.552
NLOS_cluster2  label=0  frames= 3853  features=3328  mean |H|=1.363
NLOS_cluster3  label=0  frames= 3857  features=3328  mean |H|=1.570
LOS_cluster3   label=1  frames= 7911  features=3328  mean |H|=1.531
LOS_cluster4   label=1  frames= 7900  features=3328  mean |H|=1.632
NLOS_cluster4  label=0  frames= 3856  features=3328  mean |H|=1.477
NLOS_cluster5  label=0  frames= 3862  features=3328  mean |H|=1.582


## Step 9: Save each cluster separately

The features of each cluster are saved **before the z-score**, one file per cluster:

```
clusters/LOS_cluster1_features.npy    (7939, 3328)
...
clusters/NLOS_cluster5_features.npy   (3862, 3328)
clusters/LOS_cluster1_csi.npy         (7939, 64, 52) complex64   (if SAVE_CSI)
clusters/index.json                   label, frame count and file name for each cluster
```

**Why before the z-score:** with leave-one-pair-out evaluation (20 folds), each fold has different training clusters, so the mean and std must be refitted on each fold's training clusters. Saving the features unscaled allows that without re-reading the raw files. The normalisation from Step 6 is already applied, because it works within each frame (per antenna or per frame) and does not leak information.

In [9]:
index = {}
for c, feats in per_cluster.items():
    fn = f"{c}_features.npy"
    np.save(os.path.join(CLUSTER_DIR, fn), feats.astype(np.float32))
    index[c] = {"label": label_of(c), "frames": int(len(feats)), "features_file": fn,
                "csi_file": f"{c}_csi.npy" if SAVE_CSI else None,
                "mean_amplitude_before_norm": raw_level[c]}

with open(os.path.join(CLUSTER_DIR, "index.json"), "w") as f:
    json.dump({"feature": FEATURE, "norm": NORM, "zscore_applied": False,
               "clusters": index}, f, indent=2)

for fn in sorted(os.listdir(CLUSTER_DIR)):
    print(f"{fn:32s} {os.path.getsize(os.path.join(CLUSTER_DIR, fn)) / 1e6:8.2f} MB")

LOS_cluster1_csi.npy               211.37 MB
LOS_cluster1_features.npy          105.68 MB
LOS_cluster2_csi.npy               211.87 MB
LOS_cluster2_features.npy          105.94 MB
LOS_cluster3_csi.npy               210.62 MB
LOS_cluster3_features.npy          105.31 MB
LOS_cluster4_csi.npy               210.33 MB
LOS_cluster4_features.npy          105.16 MB
NLOS_cluster1_csi.npy              156.55 MB
NLOS_cluster1_features.npy          78.27 MB
NLOS_cluster2_csi.npy              102.58 MB
NLOS_cluster2_features.npy          51.29 MB
NLOS_cluster3_csi.npy              102.69 MB
NLOS_cluster3_features.npy          51.34 MB
NLOS_cluster4_csi.npy              102.66 MB
NLOS_cluster4_features.npy          51.33 MB
NLOS_cluster5_csi.npy              102.82 MB
NLOS_cluster5_features.npy          51.41 MB
index.json                           0.00 MB


## Step 10: z-score with training-only statistics

The mean and std of each feature are computed **only from the training clusters**. They are then applied to every cluster, so no information from the test set enters preprocessing.

$\hat{x} = \dfrac{x - \mu_{train}}{\sigma_{train} + 10^{-8}}$

In [10]:
train_frames = np.concatenate([per_cluster[c] for c in TRAIN_CLUSTERS])
mu = train_frames.mean(axis=0)
sigma = train_frames.std(axis=0) + 1e-8

for c in per_cluster:
    per_cluster[c] = ((per_cluster[c] - mu) / sigma).astype(np.float32)

scaled = np.concatenate([per_cluster[c] for c in TRAIN_CLUSTERS])
print("train mean/std after scaling:", round(float(scaled.mean()), 4), round(float(scaled.std()), 4))
del train_frames, scaled

train mean/std after scaling: -0.0 1.0


## Step 11: Frame-level datasets (SVM / RF)

Each frame is one sample, labelled LOS = 1 or NLOS = 0. Each sample also keeps its cluster name, in `groups`.

The `X` arrays are written straight into `.npy` files on disk (`open_memmap`) instead of being built in memory first. This matters at full resolution, where the sequence files are several GB.

In [11]:
def make_windows(x, window=WINDOW, stride=STRIDE):
    n = (len(x) - window) // stride + 1
    if n <= 0:
        return np.empty((0, window, x.shape[1]), dtype=x.dtype)
    idx = np.arange(window)[None, :] + stride * np.arange(n)[:, None]
    return x[idx]

def n_samples(n_frames, window=None, stride=None):
    if not window:
        return n_frames
    return max((n_frames - window) // stride + 1, 0)

def stack_to_disk(name, clusters, window=None, stride=None):
    # write X straight to OUT_DIR/X_<name>.npy; return (X memmap, y, groups)
    d = next(iter(per_cluster.values())).shape[1]
    total = sum(n_samples(len(per_cluster[c]), window, stride) for c in clusters)
    shape = (total, window, d) if window else (total, d)
    X = np.lib.format.open_memmap(os.path.join(OUT_DIR, f"X_{name}.npy"),
                                  mode="w+", dtype=np.float32, shape=shape)
    ys, gs, pos = [], [], 0
    for c in clusters:
        x = per_cluster[c]
        n = n_samples(len(x), window, stride)
        if window:
            for i in range(n):
                X[pos + i] = x[i * stride:i * stride + window]
        else:
            X[pos:pos + n] = x
        ys.append(np.full(n, label_of(c), dtype=np.int8))
        gs.append(np.full(n, c))
        pos += n
    X.flush()
    return X, np.concatenate(ys), np.concatenate(gs)

X_train_f, y_train_f, g_train_f = stack_to_disk("train_frames", TRAIN_CLUSTERS)
X_test_f, y_test_f, g_test_f = stack_to_disk("test_frames", TEST_CLUSTERS)

print("train frames:", X_train_f.shape, " LOS =", int(y_train_f.sum()), " NLOS =", int((y_train_f == 0).sum()))
print("test  frames:", X_test_f.shape, " LOS =", int(y_test_f.sum()), " NLOS =", int((y_test_f == 0).sum()))

train frames: (29487, 3328)  LOS = 15897  NLOS = 13590
test  frames: (23529, 3328)  LOS = 15811  NLOS = 7718


## Step 12: Sliding-window sequences (CNN-LSTM / CNN-Transformer)

Windows are cut **within each cluster**, so no window spans two clusters.

$\text{windows per cluster} = \left\lfloor \dfrac{N - 50}{10} \right\rfloor + 1$

In [12]:
X_train_s, y_train_s, g_train_s = stack_to_disk("train_seq", TRAIN_CLUSTERS, WINDOW, STRIDE)
X_test_s, y_test_s, g_test_s = stack_to_disk("test_seq", TEST_CLUSTERS, WINDOW, STRIDE)

print("train sequences:", X_train_s.shape, " LOS =", int(y_train_s.sum()), " NLOS =", int((y_train_s == 0).sum()))
print("test  sequences:", X_test_s.shape, " LOS =", int(y_test_s.sum()), " NLOS =", int((y_test_s == 0).sum()))

train sequences: (2926, 50, 3328)  LOS = 1580  NLOS = 1346
test  sequences: (2336, 50, 3328)  LOS = 1573  NLOS = 763


## Step 13: Save outputs

The `X_*.npy` files were already written in Steps 11–12. This step saves the labels, cluster names, scaler and settings.

In [13]:
del X_train_f, X_test_f, X_train_s, X_test_s     # close the memory-mapped files

arrays = {
    "y_train_frames": y_train_f, "groups_train_frames": g_train_f,
    "y_test_frames": y_test_f,   "groups_test_frames": g_test_f,
    "y_train_seq": y_train_s,    "groups_train_seq": g_train_s,
    "y_test_seq": y_test_s,      "groups_test_seq": g_test_s,
}
for name, arr in arrays.items():
    np.save(os.path.join(OUT_DIR, f"{name}.npy"), arr)

np.savez(os.path.join(OUT_DIR, "scaler.npz"), mean=mu, std=sigma)

with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump({
        "source": "raw_traces/Pilot_Samples -> LTS FFT -> 52 subcarriers x 64 antennas, calibrated",
        "feature": FEATURE,
        "norm": NORM,
        "window": WINDOW,
        "stride": STRIDE,
        "skip_first_frames": SKIP_FIRST,
        "n_features": int(next(iter(per_cluster.values())).shape[1]),
        "save_csi": SAVE_CSI,
        "train_clusters": TRAIN_CLUSTERS,
        "test_clusters": TEST_CLUSTERS,
        "labels": {"LOS": 1, "NLOS": 0},
        "mean_amplitude_before_norm": raw_level,
    }, f, indent=2)

for fn in sorted(os.listdir(OUT_DIR)):
    path = os.path.join(OUT_DIR, fn)
    if os.path.isfile(path):
        print(f"{fn:28s} {os.path.getsize(path) / 1e6:9.2f} MB")

X_test_frames.npy               313.22 MB
X_test_seq.npy                 1554.84 MB
X_train_frames.npy              392.53 MB
X_train_seq.npy                1947.55 MB
config.json                       0.00 MB
groups_test_frames.npy            1.22 MB
groups_test_seq.npy               0.12 MB
groups_train_frames.npy           1.53 MB
groups_train_seq.npy              0.15 MB
scaler.npz                        0.03 MB
y_test_frames.npy                 0.02 MB
y_test_seq.npy                    0.00 MB
y_train_frames.npy                0.03 MB
y_train_seq.npy                   0.00 MB


## Step 14: Final checks

1. Confirm that all saved values are finite and count the sequences in each test cluster.
2. **Optional:** if `channel_data/` is available, compare the CSI extracted here with the dataset authors' processed CSI. The difference should be about 1e-6 (float32 rounding).

**Loading large files later:** use `np.load(path, mmap_mode="r")`. The data then stays on disk and is read only when it is used, so the full-resolution files never have to fit in RAM at once.

**Validation for the DL models:** do not use a random 80:20 split on the sequences. Neighbouring windows overlap by 40 frames, so validation windows would nearly duplicate training windows. Hold out whole training clusters instead, using `groups_train_seq.npy`.

In [14]:
def all_finite(arr, chunk=200):
    return all(np.isfinite(arr[i:i + chunk]).all() for i in range(0, len(arr), chunk))

for split in ["train", "test"]:
    X = np.load(os.path.join(OUT_DIR, f"X_{split}_frames.npy"), mmap_mode="r")
    S = np.load(os.path.join(OUT_DIR, f"X_{split}_seq.npy"), mmap_mode="r")
    print(f"{split}: frames {X.shape} finite={all_finite(X)}  mean={X.mean():.3f}  std={X.std():.3f}  "
          f"| seq {S.shape} finite={all_finite(S)}")

g = np.load(os.path.join(OUT_DIR, "groups_test_seq.npy"))
print(dict(zip(*np.unique(g, return_counts=True))))

# per-cluster files
with open(os.path.join(CLUSTER_DIR, "index.json")) as f:
    idx = json.load(f)["clusters"]
for c, info in idx.items():
    x = np.load(os.path.join(CLUSTER_DIR, info["features_file"]), mmap_mode="r")
    assert x.shape[0] == info["frames"]
    if info["csi_file"]:
        h = np.load(os.path.join(CLUSTER_DIR, info["csi_file"]), mmap_mode="r")
        assert h.shape == (info["frames"], 64, 52)
print("per-cluster files OK:", len(idx), "clusters")

train: frames (29487, 3328) finite=True  mean=-0.000  std=1.000  | seq (2926, 50, 3328) finite=True
test: frames (23529, 3328) finite=True  mean=0.007  std=0.725  | seq (2336, 50, 3328) finite=True
{np.str_('LOS_cluster3'): np.int64(787), np.str_('LOS_cluster4'): np.int64(786), np.str_('NLOS_cluster4'): np.int64(381), np.str_('NLOS_cluster5'): np.int64(382)}
per-cluster files OK: 9 clusters


In [15]:
REF = os.path.join(os.path.dirname(RAW_DIR), "channel_data", "processed_channe14_NLOS_cluster2.hdf5")
if os.path.exists(REF):
    H = extract_csi(file_of("NLOS_cluster2"), "NLOS_cluster2")
    with h5py.File(REF, "r") as f:
        ref = f["mob_csi"][:].transpose(1, 0, 2)
    print("shapes:", H.shape, ref.shape)
    print("max |difference| vs channel_data:", np.abs(H - ref).max())
else:
    print("channel_data not found -- skipping comparison")

shapes: (3853, 64, 52) (3853, 64, 52)
max |difference| vs channel_data: 3.960336571390643e-06
